# Text Generation with RNN, LSTM & GRU -- Adil Islam 


### Pipeline
```
Raw Text Corpus
     down
Vocabulary and char<->index mappings
     down
Sliding window -> (X, y) sequence pairs
     down
Vanilla RNN  ->  train  ->  generate text
LSTM         ->  train  ->  generate text
GRU          ->  train  ->  generate text
     down
Compare training curves, generation quality, and trade-offs
```


## 1. Install & Import


In [ ]:
# install dependencies
!pip install -q tensorflow numpy matplotlib requests


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import requests
import random
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LambdaCallback
# reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)
print(f"TensorFlow : {tf.__version__}")
print(f"GPU found  : {len(tf.config.list_physical_devices('GPU')) > 0}")
print('ready to go')


## 2. Load the Text Corpus
Using Karpathy's TinyShakespeare corpus.
It contains about one million characters from Shakespeare plays.


In [ ]:
# download text corpus
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
print('downloading corpus...')
response = requests.get(url)
full_text = response.text
USE_FULL = False
text = full_text if USE_FULL else full_text[:200000]
print(f"total chars in full corpus : {len(full_text):,}")
print(f"chars we are using       : {len(text):,}")
print('--- first 300 characters ---')
print(text[:300])


## 3. Explore the Corpus


In [ ]:
from collections import Counter
# corpus stats
all_chars = sorted(set(text))
vocab_size = len(all_chars)
char_counts = Counter(text)
print(f"unique characters  : {vocab_size}")
print(f"most common chars  : {char_counts.most_common(10)}")
print(f"full vocabulary    : {repr(''.join(all_chars))}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
common_30 = char_counts.most_common(30)
chars_30 = [repr(c) for c, _ in common_30]
freqs_30 = [count for _, count in common_30]
axes[0].bar(range(30), freqs_30, color='steelblue', edgecolor='white')
axes[0].set_xticks(range(30))
axes[0].set_xticklabels(chars_30, rotation=60, fontsize=8)
axes[0].set_title('Top 30 Most Frequent Characters', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Count')
words = text.split()
word_lengths = [len(w) for w in words]
axes[1].hist(word_lengths, bins=20, color='darkorange', edgecolor='white', alpha=0.85)
axes[1].set_title('Word Length Distribution', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Word Length')
axes[1].set_ylabel('Count')
axes[1].axvline(np.mean(word_lengths), color='red', linestyle='--', label=f'mean = {np.mean(word_lengths):.1f}')
axes[1].legend()
plt.suptitle('Corpus Analysis - TinyShakespeare', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"total words     : {len(words):,}")
print(f"unique words    : {len(set(words)):,}")
print(f"avg word length : {np.mean(word_lengths):.2f} chars")


## 4. Preprocessing - Turn Text into Numbers
The model needs integer inputs, so we map characters to indices.


In [ ]:
# build lookup tables
char2idx = {ch: i for i, ch in enumerate(all_chars)}
idx2char = {i: ch for i, ch in enumerate(all_chars)}
print(f"char2idx sample: { {k: char2idx[k] for k in list(char2idx)[:8]} }")
print(f"vocab size      : {vocab_size}")
text_as_ints = np.array([char2idx[c] for c in text], dtype=np.int32)
print(f"encoded text shape: {text_as_ints.shape}")
print(f"first 20 ints     : {text_as_ints[:20]}")
print(f"decoded back      : {''.join([idx2char[i] for i in text_as_ints[:20]])}")


In [ ]:
# build sequences
SEQ_LEN = 40
STEP = 3
sequences = []
next_chars = []
for i in range(0, len(text_as_ints) - SEQ_LEN, STEP):
    sequences.append(text_as_ints[i:i+SEQ_LEN])
    next_chars.append(text_as_ints[i+SEQ_LEN])
print(f"total training samples : {len(sequences):,}")
print(f"sequence length        : {SEQ_LEN}")
print(f"sample X[0] (decoded)  : {''.join([idx2char[i] for i in sequences[0]])}")
print(f"corresponding y[0]     : {repr(idx2char[next_chars[0]])}")


In [ ]:
# one-hot encode and split
X = np.zeros((len(sequences), SEQ_LEN, vocab_size), dtype=np.float32)
y = np.zeros((len(sequences), vocab_size), dtype=np.float32)
for i, (seq, target) in enumerate(zip(sequences, next_chars)):
    for t, char_idx in enumerate(seq):
        X[i, t, char_idx] = 1.0
    y[i, target] = 1.0
print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"memory  : {(X.nbytes + y.nbytes) / 1e6:.1f} MB")
split_idx = int(len(X) * 0.9)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]
print(f"train samples : {len(X_train):,}")
print(f"val samples   : {len(X_val):,}")


## 5. Text Generation with Temperature Sampling
Temperature controls diversity during generation.


In [ ]:
def sample_with_temperature(predictions, temperature=1.0):
    predictions = np.asarray(predictions).astype('float64')
    predictions = np.log(predictions + 1e-8) / temperature
    exp_preds = np.exp(predictions - np.max(predictions))
    predictions = exp_preds / exp_preds.sum()
    probas = np.random.multinomial(1, predictions, 1)
    return np.argmax(probas)

def generate_text(model, seed_text, num_chars=300, temperature=0.8):
    seed_text = seed_text[-SEQ_LEN:]
    seed_text = ''.join([c for c in seed_text if c in char2idx])
    if len(seed_text) < SEQ_LEN:
        seed_text = ' ' * (SEQ_LEN - len(seed_text)) + seed_text
    generated = seed_text
    current_seq = seed_text
    for _ in range(num_chars):
        x_pred = np.zeros((1, SEQ_LEN, vocab_size), dtype=np.float32)
        for t, char in enumerate(current_seq[-SEQ_LEN:]):
            if char in char2idx:
                x_pred[0, t, char2idx[char]] = 1.0
        preds = model.predict(x_pred, verbose=0)[0]
        next_idx = sample_with_temperature(preds, temperature)
        next_char = idx2char[next_idx]
        generated += next_char
        current_seq = current_seq[1:] + next_char
    return generated[SEQ_LEN:]
SEED = text[:SEQ_LEN]
print(f"seed text we'll use for all generation demos:")
print(f"'{SEED}'")


## 6. Vanilla RNN (SimpleRNN)
SimpleRNN processes one character at a time and returns the final state.


In [ ]:
def build_rnn(units=128):
    model = keras.Sequential([
        layers.Input(shape=(SEQ_LEN, vocab_size)),
        layers.SimpleRNN(units, activation='tanh'),
        layers.Dropout(0.2),
        layers.Dense(vocab_size, activation='softmax')
    ], name='VanillaRNN')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.005), loss='categorical_crossentropy', metrics=['accuracy'])
    return model
rnn_model = build_rnn(units=128)
rnn_model.summary()
print(f"total params: {rnn_model.count_params():,}")


In [ ]:
EPOCHS = 30
BATCH_SIZE = 256
def on_epoch_end_rnn(epoch, logs):
    if (epoch + 1) % 5 == 0:
        sample = generate_text(rnn_model, SEED, num_chars=150, temperature=0.8)
        print(f"\n[RNN epoch {epoch+1}] generated sample:")
        print(f"  {repr(sample[:100])}")
callbacks_rnn = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    LambdaCallback(on_epoch_end=on_epoch_end_rnn)
]
print('training vanilla RNN...')
hist_rnn = rnn_model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(X_val, y_val), callbacks=callbacks_rnn, verbose=1)
loss_rnn, acc_rnn = rnn_model.evaluate(X_val, y_val, verbose=0)
print(f"\nRNN final -> val loss: {loss_rnn:.4f} | val accuracy: {acc_rnn:.4f}")


## 7. LSTM - Long Short-Term Memory
LSTM uses gates to preserve information over longer sequences.


In [ ]:
def build_lstm(units=128):
    model = keras.Sequential([
        layers.Input(shape=(SEQ_LEN, vocab_size)),
        layers.LSTM(units),
        layers.Dropout(0.2),
        layers.Dense(vocab_size, activation='softmax')
    ], name='LSTM_Model')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.005), loss='categorical_crossentropy', metrics=['accuracy'])
    return model
lstm_model = build_lstm(units=128)
lstm_model.summary()
print(f"\nLSTM params : {lstm_model.count_params():,}")
print(f"RNN params  : {rnn_model.count_params():,}")
print(f"param ratio : {lstm_model.count_params() / rnn_model.count_params():.1f}x")


In [ ]:
def on_epoch_end_lstm(epoch, logs):
    if (epoch + 1) % 5 == 0:
        sample = generate_text(lstm_model, SEED, num_chars=150, temperature=0.8)
        print(f"\n[LSTM epoch {epoch+1}] generated sample:")
        print(f"  {repr(sample[:100])}")
callbacks_lstm = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    LambdaCallback(on_epoch_end=on_epoch_end_lstm)
]
print('training LSTM...')
hist_lstm = lstm_model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(X_val, y_val), callbacks=callbacks_lstm, verbose=1)
loss_lstm, acc_lstm = lstm_model.evaluate(X_val, y_val, verbose=0)
print(f"\nLSTM final -> val loss: {loss_lstm:.4f} | val accuracy: {acc_lstm:.4f}")


## 8. GRU - Gated Recurrent Unit
GRU is similar to LSTM but uses fewer gates and parameters.


In [ ]:
def build_gru(units=128):
    model = keras.Sequential([
        layers.Input(shape=(SEQ_LEN, vocab_size)),
        layers.GRU(units),
        layers.Dropout(0.2),
        layers.Dense(vocab_size, activation='softmax')
    ], name='GRU_Model')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.005), loss='categorical_crossentropy', metrics=['accuracy'])
    return model
gru_model = build_gru(units=128)
gru_model.summary()
print(f"\nGRU params  : {gru_model.count_params():,}")
print(f"LSTM params : {lstm_model.count_params():,}")
print(f"RNN params  : {rnn_model.count_params():,}")
print(f"GRU is {gru_model.count_params() / rnn_model.count_params():.1f}x more params than RNN")
print(f"GRU is {lstm_model.count_params() / gru_model.count_params():.2f}x fewer params than LSTM")


In [ ]:
def on_epoch_end_gru(epoch, logs):
    if (epoch + 1) % 5 == 0:
        sample = generate_text(gru_model, SEED, num_chars=150, temperature=0.8)
        print(f"\n[GRU epoch {epoch+1}] generated sample:")
        print(f"  {repr(sample[:100])}")
callbacks_gru = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    LambdaCallback(on_epoch_end=on_epoch_end_gru)
]
print('training GRU...')
hist_gru = gru_model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(X_val, y_val), callbacks=callbacks_gru, verbose=1)
loss_gru, acc_gru = gru_model.evaluate(X_val, y_val, verbose=0)
print(f"\nGRU final -> val loss: {loss_gru:.4f} | val accuracy: {acc_gru:.4f}")


## 9. Training Curves - All Three Models


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
model_data = [(hist_rnn, 'Vanilla RNN', '#e74c3c'), (hist_lstm, 'LSTM', '#2ecc71'), (hist_gru, 'GRU', '#3498db')]
for col, (hist, name, color) in enumerate(model_data):
    ep = range(1, len(hist.history['loss']) + 1)
    axes[0, col].plot(ep, hist.history['loss'], color=color, linewidth=2, label='train')
    axes[0, col].plot(ep, hist.history['val_loss'], color=color, linewidth=2, linestyle='--', label='val', alpha=0.7)
    axes[0, col].set_title(f'{name} - Loss', fontweight='bold', fontsize=12)
    axes[0, col].set_xlabel('Epoch')
    axes[0, col].set_ylabel('Categorical Cross-Entropy')
    axes[0, col].legend()
    axes[1, col].plot(ep, hist.history['accuracy'], color=color, linewidth=2, label='train')
    axes[1, col].plot(ep, hist.history['val_accuracy'], color=color, linewidth=2, linestyle='--', label='val', alpha=0.7)
    axes[1, col].set_title(f'{name} - Accuracy', fontweight='bold', fontsize=12)
    axes[1, col].set_xlabel('Epoch')
    axes[1, col].set_ylabel('Accuracy')
    axes[1, col].legend()
    axes[1, col].set_ylim(0, 0.6)
plt.suptitle('Training Curves - RNN vs LSTM vs GRU', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for hist, name, color in model_data:
    ep = range(1, len(hist.history['val_loss']) + 1)
    ax1.plot(ep, hist.history['val_loss'], color=color, linewidth=2, label=name)
    ax2.plot(ep, hist.history['val_accuracy'], color=color, linewidth=2, label=name)
ax1.set_title('Validation Loss - All Models', fontweight='bold', fontsize=12)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax2.set_title('Validation Accuracy - All Models', fontweight='bold', fontsize=12)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
plt.suptitle('Side-by-Side Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 10. Performance Comparison


In [ ]:
import pandas as pd
results = {
    'Model': ['Vanilla RNN', 'LSTM', 'GRU'],
    'Parameters': [rnn_model.count_params(), lstm_model.count_params(), gru_model.count_params()],
    'Val Loss': [round(loss_rnn, 4), round(loss_lstm, 4), round(loss_gru, 4)],
    'Val Accuracy': [round(acc_rnn, 4), round(acc_lstm, 4), round(acc_gru, 4)],
    'Epochs Trained': [len(hist_rnn.history['loss']), len(hist_lstm.history['loss']), len(hist_gru.history['loss'])],
}
df = pd.DataFrame(results)
df['Val Loss Rank'] = df['Val Loss'].rank().astype(int)
print('MODEL COMPARISON TABLE')
print('=' * 75)
print(df.to_string(index=False))
print('=' * 75)
print('lower val loss = better | higher accuracy = better')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#e74c3c', '#2ecc71', '#3498db']
ax1.bar(results['Model'], results['Val Loss'], color=colors, edgecolor='white')
ax1.set_title('Validation Loss (lower is better)', fontweight='bold')
ax1.set_ylabel('Loss')
for i, v in enumerate(results['Val Loss']):
    ax1.text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')
ax2.bar(results['Model'], results['Val Accuracy'], color=colors, edgecolor='white')
ax2.set_title('Validation Accuracy (higher is better)', fontweight='bold')
ax2.set_ylabel('Accuracy')
for i, v in enumerate(results['Val Accuracy']):
    ax2.text(i, v + 0.003, f'{v:.4f}', ha='center', fontweight='bold')
plt.suptitle('RNN vs LSTM vs GRU - Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 11. Generated Text - All Three Models
Same seed and temperature for each model.


In [ ]:
TEMPERATURE = 0.8
NUM_CHARS = 400
print('=' * 70)
print(f"SEED: '{SEED}'")
print('=' * 70)
for model, name in [(rnn_model, 'VANILLA RNN'), (lstm_model, 'LSTM'), (gru_model, 'GRU')]:
    generated = generate_text(model, SEED, num_chars=NUM_CHARS, temperature=TEMPERATURE)
    print(f"\n--- {name} (temp={TEMPERATURE}) ---")
    print(generated)
    print()


## 12. Temperature Experiment - LSTM at Different Temperatures
Test how temperature affects generation diversity.


In [ ]:
temperatures = [0.2, 0.5, 0.8, 1.0, 1.2]
print(f"SEED: '{SEED}'")
print('=' * 70)
for temp in temperatures:
    generated = generate_text(lstm_model, SEED, num_chars=200, temperature=temp)
    print(f"\n--- temperature = {temp} ---")
    if temp <= 0.3:
        print('(very conservative - safe but repetitive)')
    elif temp <= 0.7:
        print('(balanced - coherent with some variety)')
    elif temp <= 1.0:
        print('(natural - good mix of predictability and creativity)')
    else:
        print('(risky - creative but starts breaking grammar rules)')
    print(generated)


## 13.  Stacked LSTM
Two stacked LSTM layers can capture higher-level structure.


In [ ]:
def build_stacked_lstm(units=128):
    model = keras.Sequential([
        layers.Input(shape=(SEQ_LEN, vocab_size)),
        layers.LSTM(units, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(units),
        layers.Dropout(0.2),
        layers.Dense(vocab_size, activation='softmax')
    ], name='Stacked_LSTM')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.003), loss='categorical_crossentropy', metrics=['accuracy'])
    return model
stacked_lstm = build_stacked_lstm(units=128)
stacked_lstm.summary()
print(f"\nstacked LSTM params : {stacked_lstm.count_params():,}")


In [ ]:
def on_epoch_end_stacked(epoch, logs):
    if (epoch + 1) % 5 == 0:
        sample = generate_text(stacked_lstm, SEED, num_chars=150, temperature=0.8)
        print(f"\n[Stacked LSTM epoch {epoch+1}]:")
        print(f"  {repr(sample[:120])}")
callbacks_stacked = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    LambdaCallback(on_epoch_end=on_epoch_end_stacked)
]
print('training stacked LSTM...')
hist_stacked = stacked_lstm.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(X_val, y_val), callbacks=callbacks_stacked, verbose=1)
loss_stacked, acc_stacked = stacked_lstm.evaluate(X_val, y_val, verbose=0)
print(f"\nStacked LSTM -> val loss: {loss_stacked:.4f} | val accuracy: {acc_stacked:.4f}")


In [ ]:
print('--- STACKED LSTM GENERATED TEXT ---')
print(generate_text(stacked_lstm, SEED, num_chars=500, temperature=0.8))


## 14. Key Insights & Analysis
### What each model learned
All three models predict the next character from the previous 40 characters.

### Architecture comparison
| Feature | Vanilla RNN | LSTM | GRU |
|---|---|---|---|
| Hidden states | 1 | 2 | 1 |
| Gates | None | 3 | 2 |
| Params (128 units) | ~17K | ~68K | ~51K |
| Vanishing gradient | Severe | Mostly solved | Mostly solved |
| Training speed | Fastest | Slowest | Middle |
| Long-range memory | Poor | Excellent | Good |
| Best for | short patterns | long documents | faster training |

### Temperature insights
- temp=0.2: safe and repetitive
- temp=0.8: balanced and varied
- temp=1.2: creative but less coherent


In [ ]:
print('=' * 65)
print('WEEK 5 - TEXT GENERATION WITH RNN, LSTM & GRU - DONE')
print('=' * 65)
print()
final_results = {
    'Vanilla RNN': (rnn_model.count_params(), loss_rnn, acc_rnn),
    'LSTM': (lstm_model.count_params(), loss_lstm, acc_lstm),
    'GRU': (gru_model.count_params(), loss_gru, acc_gru),
    'Stacked LSTM': (stacked_lstm.count_params(), loss_stacked, acc_stacked),
}
print(f"{'Model':<20} {'Params':>10} {'Val Loss':>10} {'Val Acc':>10}")
print(' ' + '-'*55)
for name, (params, loss, acc) in final_results.items():
    print(f"{name:<20} {params:>10,} {loss:>10.4f} {acc:>10.4f}")
print()
print('  Corpus      : TinyShakespeare (~200K chars)')
print('  Task        : Character-level next-char prediction')
print('  Sequence len: 40 chars')
print('  Vocab size  : ~65 unique characters')
print()
print('  Author : Adil Islam')
print('  Track  : Celebal Technologies - Data Science Internship (CEI)')
print('=' * 65)
